# Helmet and Number Plate Detection using YOLO (Kaggle version)

Detects four classes in a traffic image:

| Class ID | Name |
|---|---|
| 0 | with helmet |
| 1 | without helmet |
| 2 | rider |
| 3 | number plate |

**Model:** YOLOv8s (Ultralytics), fine-tuned on the Kaggle dataset *Rider With Helmet Without Helmet Number Plate*.

**Before running, set up the Kaggle notebook:**
1. Right side panel > **Session options > Accelerator** > choose **GPU T4 x2** or **GPU P100**.
2. Right side panel > **Session options > Internet** > turn **On** (needed to download `yolov8s.pt`). Kaggle may ask for phone verification.
3. Click **+ Add Input** > search `rider with helmet without helmet number plate` (by aneesarom) > **Add**.

**Outline**
1. Install and import libraries
2. Find and copy the dataset
3. Create the dataset config file
4. Train the model
5. Check training results
6. Test on validation images
7. Count helmets and violations
8. Read the number plate text (OCR)
9. Save the trained model

## 1. Install and import libraries

In [ ]:
!pip install -q ultralytics easyocr

In [ ]:
import os
import glob
import shutil
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

## 2. Find and copy the dataset
Files in `/kaggle/input` are read-only, so the dataset is copied to `/kaggle/working`. The code searches for the folder that contains `train/images`.

In [ ]:
root = None
for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
    if os.path.isdir(os.path.join(dirpath, 'train', 'images')):
        root = dirpath
        break

print('Dataset found at:', root)
assert root is not None, 'Dataset not found. Add it using + Add Input.'

DATA_DIR = '/kaggle/working/dataset'
if not os.path.exists(DATA_DIR):
    shutil.copytree(root, DATA_DIR)

for split in ['train', 'val']:
    for sub in ['images', 'labels']:
        p = os.path.join(DATA_DIR, split, sub)
        n = len(os.listdir(p)) if os.path.isdir(p) else 'MISSING'
        print(split, sub, n)

## 3. Create the dataset config file
This YAML file tells YOLO where the images are and what the class names are. The class order matches the trained model (0 to 3).

In [ ]:
yaml_text = f"""path: {DATA_DIR}
train: train/images
val: val/images
nc: 4
names: ['with helmet', 'without helmet', 'rider', 'number plate']
"""
with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_text)
print(yaml_text)

## 4. Train the model
Fine-tunes the pretrained `yolov8s.pt` for 10 epochs. On a Kaggle GPU this takes a few minutes.

In [ ]:
model = YOLO('yolov8s.pt')

model.train(
    data='/kaggle/working/data.yaml',
    epochs=10,
    imgsz=640,
    project='/kaggle/working/training_results',
    name='results',
    exist_ok=True
)

## 5. Check training results
Loss curves and the confusion matrix are saved in the results folder.

In [ ]:
save_dir = '/kaggle/working/training_results/results'
for name in ['results.png', 'confusion_matrix.png']:
    path = os.path.join(save_dir, name)
    if os.path.exists(path):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(name)
        plt.show()

## 6. Test on validation images
Load the best model from training and run it on a few images.

In [ ]:
best = YOLO('/kaggle/working/training_results/results/weights/best.pt')
print(best.names)

def detect_and_show(image_path, conf=0.25):
    result = best(image_path, conf=conf)[0]
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 7))
    plt.imshow(annotated)
    plt.axis('off')
    plt.show()
    return result

val_images = sorted(glob.glob(os.path.join(DATA_DIR, 'val', 'images', '*')))
for p in val_images[:3]:
    last = detect_and_show(p)

To test **your own image** on Kaggle: click **+ Add Input > Upload**, create a small dataset with your photo, then use its path, for example
`/kaggle/input/my-photos/myphoto.jpg`.

In [ ]:
# result = detect_and_show('/kaggle/input/my-photos/myphoto.jpg')

## 7. Count helmets and violations

In [ ]:
def summarize(result):
    counts = {}
    for cls_id in result.boxes.cls.tolist():
        name = best.names[int(cls_id)]
        counts[name] = counts.get(name, 0) + 1
    return counts

counts = summarize(last)
print(counts)
if counts.get('without helmet', 0) > 0:
    print('Violation: rider without helmet detected')
else:
    print('No helmet violation detected')

## 8. Read the number plate text (OCR)
The plate box is cropped and passed to EasyOCR. This is an extra step and works best on clear, close-up plates. It needs Internet turned on to download the OCR model.

In [ ]:
import easyocr
reader = easyocr.Reader(['en'], gpu=True)

def read_plates(image_path):
    img = cv2.imread(image_path)
    res = best(image_path)[0]
    texts = []
    for box in res.boxes:
        if best.names[int(box.cls)] == 'number plate':
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            crop = img[y1:y2, x1:x2]
            out = reader.readtext(crop)
            texts.append(' '.join(t[1] for t in out))
    return texts

for p in val_images[:3]:
    print(os.path.basename(p), '->', read_plates(p))

## 9. Save the trained model
The trained weights are in `/kaggle/working/training_results/results/weights/best.pt`. After the notebook finishes, download `best.pt` from the **Output** tab on the right to use it on your laptop.

In [ ]:
shutil.copy('/kaggle/working/training_results/results/weights/best.pt', '/kaggle/working/best.pt')
print('Saved: /kaggle/working/best.pt')

## Conclusion
The model detects riders, helmets, missing helmets and number plates. Accuracy is better on clear, close images and lower in crowded scenes with small riders. It could be improved with more epochs, more data, and a larger model.